# 02 - MiniCNN 基线模型训练

模型：MiniCNN（3 层卷积基线；参数量以 model_spec 输出为准）
目标：建立性能基线，验证数据pipeline正确性

控制方式：
- **开始训练**: 运行下方训练Cell
- **暂停训练**: 点击 Jupyter 停止按钮(■)，断点自动保存
- **继续训练**: 运行下方Checkpoint选择Cell，选择断点后运行训练Cell

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from data.dataloader import create_dataloaders
from training.checkpoint import select_checkpoint_widget
from training.trainer import (
    PROJECT_ROOT,
    Trainer,
    load_config,
    set_seed,
)
from utils.model_spec import build_model_from_spec, make_spec_from_config

# 加载配置（确定性/性能模式由配置 cudnn_deterministic 决定）
config = load_config()
set_seed(config['seed'], deterministic=config['training'].get('cudnn_deterministic', False))

MODEL_NAME = 'mini_cnn'
model_config = config['models'][MODEL_NAME]
CLASS_NAMES = config['data']['class_names']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"设备: {device}")
print(f"模型: {MODEL_NAME}")
print(f"配置: {model_config}")


## 1. Checkpoint 选择（续训时使用）

In [ ]:
# 选择 Checkpoint：点击「确认选择」后运行下一个 Cell
# - 列出 training/runs/ 下该模型的可续训断点（新格式 run：last / best / 中断断点）
# - 选择结果自动写入 RESUME_FROM；也可手动设置 RESUME_FROM = None（从头训练）或路径
select_checkpoint_widget(MODEL_NAME)


## 2. 准备数据 & 构建模型

In [ ]:
# RESUME_FROM 由上方控件设置（或手动指定路径 / None）
RESUME_FROM = globals().get('RESUME_FROM', None)

resume_path = None
run_dir = None
if RESUME_FROM:
    resume_path = Path(RESUME_FROM)
    # 精确恢复：加载时校验配置/数据管线与断点一致（不满足将明确拒绝，S01）
    meta = Trainer.load_checkpoint_metadata(resume_path)
    print(f"恢复训练: {resume_path}")
    if meta:
        print(f"  run={meta['run_id']} epoch={meta['epoch']} "
              f"val_acc={meta['val_acc']:.4f} partial={meta['partial']}")
    run_dir = resume_path.parent.parent
else:
    print("从头开始训练（将创建新的 run 目录）")

# 数据（在应用恢复条件之后再构建 DataLoader）
train_loader, val_loader, _, _ = create_dataloaders(config, MODEL_NAME, include_test=False)
print(f"训练集: {len(train_loader.dataset)}, 验证集: {len(val_loader.dataset)}")
print("PrivateTest: 未构建（评估时按需加载）")

# 模型：统一构造入口（与训练保存 / 离线评估 / 推理应用共用同一 model_spec）
spec = make_spec_from_config(config, MODEL_NAME)
model = build_model_from_spec(spec)
print(f"model_spec: {spec.display_summary()}")

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=None,
    config=config,
    model_name=MODEL_NAME,
    device=device,
    run_dir=run_dir,
    run_meta_extra={"cli_args": {"source": "notebook"}},
)

if resume_path is not None:
    trainer.load_checkpoint(resume_path)

print(f"总参数量: {trainer.total_params:,}")
print(f"Run 目录: {trainer.run_dir}")


## 3. 性能诊断（可选）

> 运行此 Cell 检查训练速度是否正常。如果速度异常低，会自动给出建议。

In [ ]:
# 性能诊断：逐步计时，定位瓶颈（会执行真实训练步骤，仅作性能参考）
trainer.diagnose(num_steps=5)


In [ ]:
# fit(N) = 本次训练 N 轮（开放式；可反复调用续训）
# 每个 epoch 结束自动保存 last.pth；best.pth 受配置 monitor_metric / save_best 控制
history = trainer.fit(60)


## 4. 训练曲线

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], 'b-', label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'], 'r-', label='Val Loss')
axes[0].set_title(f'{MODEL_NAME} - Loss Curve')
axes[0].legend()

axes[1].plot(epochs_range, history['train_acc'], 'b-', label='Train Acc')
axes[1].plot(epochs_range, history['val_acc'], 'r-', label='Val Acc')
axes[1].set_title(f'{MODEL_NAME} - Accuracy Curve')
axes[1].legend()

axes[2].plot(epochs_range, history['lr'], 'g-')
axes[2].set_title(f'{MODEL_NAME} - LR Schedule')

plt.tight_layout()
curve_dir = PROJECT_ROOT / 'analysis' / 'training_curves'
curve_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(curve_dir / f'{MODEL_NAME}_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. 测试集评估（统一评估入口）（统一评估入口）

In [ ]:
# 测试集评估：统一评估入口（显式指定 checkpoint 与 split；无内存状态依赖）
# 说明：决策/调参阶段使用 PublicTest（与比较协议一致）；
#      方案冻结后的最终评估再改为 SPLIT = 'PublicTest'（仅评估一次）
# 评估产物（完整指标 + 逐样本预测 + 权重/划分哈希）保存到 analysis/evaluations/
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report

from utils.evaluation import evaluate_checkpoint

SPLIT = 'PublicTest'
ckpt_path = trainer.checkpoints_dir / ('best.pth' if trainer.save_best else 'last.pth')
result = evaluate_checkpoint(ckpt_path, split=SPLIT, device='cpu')

preds = np.array(result['predictions']['preds'])
labels = np.array(result['predictions']['labels'])
probs = np.array(result['predictions']['probs'])
all_preds, all_labels, all_probs = preds, labels, probs  # 供 ROC Cell 使用

print("分类报告:")
print(classification_report(labels, preds, target_names=CLASS_NAMES, zero_division=0))
print(f"accuracy={result['metrics']['accuracy']:.4f} | "
      f"macro_f1={result['metrics']['macro_f1']:.4f} | "
      f"balanced_acc={result['metrics']['balanced_accuracy']:.4f} | "
      f"ECE={result['metrics']['ece']:.4f}")

# 混淆矩阵
cm = np.array(result['metrics']['confusion_matrix'])
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'{MODEL_NAME} - Confusion Matrix ({SPLIT})')
plt.tight_layout()
cm_dir = PROJECT_ROOT / 'analysis' / 'confusion_matrices'
cm_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(cm_dir / f'{MODEL_NAME}_cm.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. ROC 曲线

In [ ]:
# ROC 曲线 — 依赖上方「测试集评估」Cell 生成的 all_labels / all_probs
from sklearn.metrics import auc, roc_curve
from sklearn.preprocessing import label_binarize

if "all_labels" not in dir() or len(all_labels) == 0:
    raise RuntimeError("请先运行上方「测试集评估」Cell，再运行此 Cell")

y_true_bin = label_binarize(all_labels, classes=list(range(len(CLASS_NAMES))))

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.Set1(np.linspace(0, 1, len(CLASS_NAMES)))

for i, (name, color) in enumerate(zip(CLASS_NAMES, colors, strict=True)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC = {roc_auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=2)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"{MODEL_NAME} - ROC Curves (One-vs-Rest, {SPLIT})")
ax.legend(loc="lower right")
plt.tight_layout()
roc_dir = PROJECT_ROOT / "analysis" / "roc_curves"
roc_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(roc_dir / f"{MODEL_NAME}_roc.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. 导出最终模型到推理目录（显式导出 + 来源清单）（显式导出 + 来源清单）

In [ ]:
# 导出最优模型到推理目录（显式导出 + 来源清单；默认不覆盖同名文件）
# 导出后可在应用中选择该权重（显示 run 来源与 SHA-256）
from tools.export_model import export_checkpoint

ckpt_to_export = trainer.checkpoints_dir / ('best.pth' if trainer.save_best else 'last.pth')
if ckpt_to_export.exists():
    try:
        entry = export_checkpoint(ckpt_to_export, force=False)
        print(f"已导出: {entry['file']}")
        print(f"  run={entry['run_id']} sha256={entry['exported_sha256'][:16]}...")
        print(f"  metrics={entry['metrics']}")
    except FileExistsError as e:
        print(f"同名文件已存在（未覆盖）: {e}")
        print("如需覆盖: export_checkpoint(ckpt_to_export, force=True)")
else:
    print(f"未找到 {ckpt_to_export}，请先训练或手动指定断点路径")


## 8. 一键导出分析图表到 analysis_saved/

In [ ]:
# 将从 §4-6 生成的训练曲线、混淆矩阵、ROC 曲线复制到
# analysis_saved/{MODEL_NAME}_{时间戳}/，文件保持原名

import shutil
from datetime import datetime

export_root = PROJECT_ROOT / 'analysis_saved'
export_root.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
export_dir = export_root / f'{MODEL_NAME}_{ts}'
export_dir.mkdir(parents=True, exist_ok=True)

sources = [
    PROJECT_ROOT / 'analysis' / 'training_curves' / f'{MODEL_NAME}_training_curves.png',
    PROJECT_ROOT / 'analysis' / 'confusion_matrices' / f'{MODEL_NAME}_cm.png',
    PROJECT_ROOT / 'analysis' / 'roc_curves' / f'{MODEL_NAME}_roc.png',
]

copied = 0
for src in sources:
    if src.exists():
        shutil.copy2(src, export_dir)
        copied += 1
        print(f'已复制: {src.name}')
    else:
        print(f'未找到: {src.name}')

print(f'\n共导出 {copied}/3 张图到: {export_dir}')
